# 03 - Silver orders (typed, validated, deduplicated)

**Input:** `bronze_orders`

**Outputs (three tables, every Bronze row lands in exactly one):**

| Table | Contains |
|---|---|
| `silver_orders` | valid rows that **won** deduplication (one row per `order_id`) |
| `quarantine_orders` | rows that failed >= 1 validation rule, with `rejection_reason` |
| `duplicate_losers_orders` | valid rows that **lost** to another valid row with the same `order_id` |

### Precedence (copy into README)

**Validate first, then deduplicate among valid records.**

1. Parse and clean every row.
2. Any row failing a rule goes to quarantine. It never competes in deduplication.
3. Only valid rows compete for each `order_id`; one deterministic winner goes to `silver_orders`.
4. The other valid rows go to `duplicate_losers_orders` and point to their winner.

### Rerun behavior

Each run **fully rebuilds** the three tables from Bronze with `CREATE OR REPLACE TABLE`, so reruns give the same result and never append twice.

### Important

The allowed lists and numeric/date limits below are **rules you choose**, not assignment-provided rules. Run the pre-flight cell (section 3) first and set the widgets from what your data really contains.

## 1. Parameters (these ARE your documented validation rules)

In [0]:
dbutils.widgets.text("catalog", "workspace", "Catalog")
dbutils.widgets.text("schema", "ecom_poc", "Schema")

# Allowed order statuses (normalized to lowercase). Verify against section 3 before trusting.
dbutils.widgets.text(
    "allowed_statuses",
    "completed,pending,shipped,cancelled,returned,refunded",
    "Allowed order statuses (comma list, lowercase)",
)
# Leave blank = do NOT enforce a list (only 'not blank' is checked)
dbutils.widgets.text("allowed_categories", "", "Allowed categories (blank = not enforced)")
dbutils.widgets.text("allowed_countries", "", "Allowed countries (blank = not enforced)")

dbutils.widgets.text("max_quantity", "1000", "Max quantity per order line")
dbutils.widgets.text("max_unit_price", "100000", "Max unit price")
dbutils.widgets.text("min_order_date", "2000-01-01", "Earliest acceptable order_date (yyyy-MM-dd)")

In [0]:
import datetime

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()


def parse_list(text, transform):
    """'a, B ,c' -> ['a','b','c'] using the given transform; blank text -> []"""
    return [transform(x.strip()) for x in text.split(",") if x.strip()]


ALLOWED_STATUSES = parse_list(dbutils.widgets.get("allowed_statuses"), str.lower)
ALLOWED_CATEGORIES = parse_list(dbutils.widgets.get("allowed_categories"), str.lower)
ALLOWED_COUNTRIES = parse_list(dbutils.widgets.get("allowed_countries"), str.upper)

MAX_QUANTITY = int(dbutils.widgets.get("max_quantity"))
MAX_UNIT_PRICE = float(dbutils.widgets.get("max_unit_price"))
MIN_ORDER_DATE = datetime.date.fromisoformat(dbutils.widgets.get("min_order_date").strip())

# "Today" is captured ONCE so the future-date rule is identical for every table written in this run
RUN_DATE = spark.sql("SELECT current_date() AS d").collect()[0]["d"]

def sql_in_list(values):
    """['a','b'] -> 'a', 'b' (quoted for a SQL IN (...) list, quotes escaped)"""
    return ", ".join("'" + v.replace("'", "''") + "'" for v in values)


assert ALLOWED_STATUSES, "allowed_statuses cannot be empty - order_status must be validated against a documented set."

# Fully qualified object names
BRONZE = f"`{CATALOG}`.`{SCHEMA}`.`bronze_orders`"
SILVER = f"`{CATALOG}`.`{SCHEMA}`.`silver_orders`"
QUARANTINE = f"`{CATALOG}`.`{SCHEMA}`.`quarantine_orders`"
DUP_LOSERS = f"`{CATALOG}`.`{SCHEMA}`.`duplicate_losers_orders`"

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

print("Run date (UTC session) :", RUN_DATE)
print("Allowed statuses       :", ALLOWED_STATUSES)
print("Allowed categories     :", ALLOWED_CATEGORIES or "not enforced (only non-blank)")
print("Allowed countries      :", ALLOWED_COUNTRIES or "not enforced (only non-blank)")
print("Quantity range         : 1 .. ", MAX_QUANTITY)
print("Unit price range       : 0 .. ", MAX_UNIT_PRICE)
print("Order date range       :", MIN_ORDER_DATE, ".. run date")

## 2. Bronze input check

Silver must read exactly one snapshot, and it must not be empty.

In [0]:
bronze_df = spark.table(BRONZE)
bronze_count = bronze_df.count()
batches = [r["_batch_id"] for r in bronze_df.select("_batch_id").distinct().collect()]

print(f"bronze_orders rows : {bronze_count:,}")
print(f"batch id(s)        : {batches}")

assert bronze_count > 0, "bronze_orders is empty - run 02_bronze_ingestion first."
assert len(batches) == 1, f"bronze_orders holds {len(batches)} batches; expected exactly 1 snapshot."

## 3. Pre-flight: what values does Bronze actually contain?

Use this to confirm the allowed lists and ranges **before** the rules are applied. Labels are shown after the same normalization Silver uses (trim + case).

In [0]:
TRIM_RE = r"^\\s+|\\s+$"   # SQL-escaped regex for leading/trailing whitespace (spaces, tabs, newlines)


def clean(col):
    """Strip leading/trailing whitespace of any kind."""
    return f"regexp_replace({col}, '{TRIM_RE}', '')"


def norm_label(col, how):
    """Trim, turn blank into NULL, then lower/upper-case."""
    return f"{how}(NULLIF({clean(col)}, ''))"


display(spark.sql(f"""
    SELECT {norm_label('order_status', 'LOWER')} AS order_status_normalized,
           COUNT(*) AS rows,
           CASE WHEN {norm_label('order_status', 'LOWER')} IN ({sql_in_list(ALLOWED_STATUSES)})
                THEN 'allowed' ELSE 'WOULD BE QUARANTINED' END AS status_rule
    FROM {BRONZE}
    GROUP BY 1, 3
    ORDER BY rows DESC
"""))

In [0]:
display(spark.sql(f"""
    SELECT {norm_label('category', 'LOWER')} AS category_normalized, COUNT(*) AS rows
    FROM {BRONZE} GROUP BY 1 ORDER BY rows DESC LIMIT 50
"""))

In [0]:
display(spark.sql(f"""
    SELECT {norm_label('country', 'UPPER')} AS country_normalized, COUNT(*) AS rows
    FROM {BRONZE} GROUP BY 1 ORDER BY rows DESC LIMIT 50
"""))

In [0]:
# Observed ranges among values that parse (helps you set max_quantity, max_unit_price, min_order_date)
display(spark.sql(f"""
    SELECT
        MIN(try_cast({clean('quantity')} AS DECIMAL(18,4)))   AS min_quantity,
        MAX(try_cast({clean('quantity')} AS DECIMAL(18,4)))   AS max_quantity,
        MIN(try_cast({clean('unit_price')} AS DECIMAL(18,2))) AS min_unit_price,
        MAX(try_cast({clean('unit_price')} AS DECIMAL(18,2))) AS max_unit_price,
        MIN(try_cast({clean('order_date')} AS DATE))          AS min_order_date,
        MAX(try_cast({clean('order_date')} AS DATE))          AS max_order_date
    FROM {BRONZE}
"""))

## 4. Step A - clean text and parse types safely

`try_cast` returns NULL on bad input instead of failing. **Every NULL produced here is caught by a validation rule in step B**, so nothing unparseable is silently accepted.

| Field | Cleaning / parsing |
|---|---|
| ids | strip whitespace; blank -> NULL; case is **kept** |
| `order_date` | must look like `yyyy-MM-dd` AND be a real calendar date, else NULL |
| `quantity` | parsed as decimal, must be a whole number, stored as `INT` |
| `unit_price` | parsed as `DECIMAL(18,2)` (values with more than 2 decimals are rounded) |
| `order_status`, `category` | strip whitespace, lowercase |
| `country` | strip whitespace, uppercase |

In [0]:
DATE_RE = "^[0-9]{4}-[0-9]{2}-[0-9]{2}$"

# Layer 1: cleaned text (raw columns are kept untouched alongside, for quarantine evidence)
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW orders_p1 AS
SELECT
    order_id, customer_id, order_date, category, quantity, unit_price, order_status, country,   -- raw strings
    _ingested_at, _source_file, _batch_id, _row_hash, _row_id,
    NULLIF({clean('order_id')}, '')     AS order_id_c,
    NULLIF({clean('customer_id')}, '')  AS customer_id_c,
    NULLIF({clean('order_date')}, '')   AS date_txt,
    NULLIF({clean('quantity')}, '')     AS qty_txt,
    NULLIF({clean('unit_price')}, '')   AS price_txt,
    {norm_label('category', 'LOWER')}      AS category_c,
    {norm_label('order_status', 'LOWER')}  AS status_c,
    {norm_label('country', 'UPPER')}       AS country_c
FROM {BRONZE}
""")

In [0]:
# Layer 2: typed values (NULL when unparseable)
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW orders_p2 AS
SELECT
    *,
    CASE WHEN date_txt RLIKE '{DATE_RE}' THEN try_cast(date_txt AS DATE) END   AS order_date_p,
    try_cast(qty_txt AS DECIMAL(18,4))                                         AS qty_num,
    CASE WHEN try_cast(qty_txt AS DECIMAL(18,4)) IS NOT NULL
          AND try_cast(qty_txt AS DECIMAL(18,4)) = floor(try_cast(qty_txt AS DECIMAL(18,4)))
          AND try_cast(qty_txt AS DECIMAL(18,4)) BETWEEN -2147483648 AND 2147483647
         THEN CAST(try_cast(qty_txt AS DECIMAL(18,4)) AS INT) END              AS quantity_p,
    try_cast(price_txt AS DECIMAL(18,2))                                       AS unit_price_p
FROM orders_p1
""")

print("Temp views orders_p1 and orders_p2 created (no data written yet).")

## 5. Step B - validation matrix

Each failed rule adds a **reason code** to the row. A row can have several reasons.

| Field | Reason codes |
|---|---|
| `order_id` | `order_id_missing` |
| `customer_id` | `customer_id_missing` |
| `order_date` | `order_date_missing`, `order_date_invalid`, `order_date_future`, `order_date_too_old` |
| `quantity` | `quantity_missing`, `quantity_not_numeric`, `quantity_not_integer`, `quantity_not_positive`, `quantity_above_max` |
| `unit_price` | `unit_price_missing`, `unit_price_not_numeric`, `unit_price_negative`, `unit_price_above_max` |
| `order_status` | `order_status_missing`, `order_status_not_allowed` |
| `category` | `category_missing`, `category_not_allowed` (only if a list is set) |
| `country` | `country_missing`, `country_not_allowed` (only if a list is set) |

In [0]:
# Category / country: "not blank" always; "in allowed list" only when a list was provided
category_rule = "CASE WHEN category_c IS NULL THEN 'category_missing'"
if ALLOWED_CATEGORIES:
    category_rule += f" WHEN category_c NOT IN ({sql_in_list(ALLOWED_CATEGORIES)}) THEN 'category_not_allowed'"
category_rule += " END"

country_rule = "CASE WHEN country_c IS NULL THEN 'country_missing'"
if ALLOWED_COUNTRIES:
    country_rule += f" WHEN country_c NOT IN ({sql_in_list(ALLOWED_COUNTRIES)}) THEN 'country_not_allowed'"
country_rule += " END"

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW orders_validated AS
SELECT
    *,
    filter(
        array(
            CASE WHEN order_id_c    IS NULL THEN 'order_id_missing' END,
            CASE WHEN customer_id_c IS NULL THEN 'customer_id_missing' END,

            CASE WHEN date_txt IS NULL                                THEN 'order_date_missing'
                 WHEN order_date_p IS NULL                            THEN 'order_date_invalid'
                 WHEN order_date_p > DATE'{RUN_DATE}'                 THEN 'order_date_future'
                 WHEN order_date_p < DATE'{MIN_ORDER_DATE}'           THEN 'order_date_too_old' END,

            CASE WHEN qty_txt IS NULL          THEN 'quantity_missing'
                 WHEN qty_num IS NULL          THEN 'quantity_not_numeric'
                 WHEN quantity_p IS NULL       THEN 'quantity_not_integer'
                 WHEN quantity_p <= 0          THEN 'quantity_not_positive'
                 WHEN quantity_p > {MAX_QUANTITY} THEN 'quantity_above_max' END,

            CASE WHEN price_txt IS NULL        THEN 'unit_price_missing'
                 WHEN unit_price_p IS NULL     THEN 'unit_price_not_numeric'
                 WHEN unit_price_p < 0         THEN 'unit_price_negative'
                 WHEN unit_price_p > {MAX_UNIT_PRICE} THEN 'unit_price_above_max' END,

            CASE WHEN status_c IS NULL         THEN 'order_status_missing'
                 WHEN status_c NOT IN ({sql_in_list(ALLOWED_STATUSES)}) THEN 'order_status_not_allowed' END,

            {category_rule},
            {country_rule}
        ),
        x -> x IS NOT NULL
    ) AS reasons
FROM orders_p2
""")

# Only VALID rows compete for deduplication. Invalid rows are handled separately (quarantine) and never get a rank.
# WINNER RULE (my choice - document it): for the same order_id among valid rows
#   1) latest order_date wins, 2) then smallest _row_hash, 3) then smallest _row_id  -> fully deterministic.
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders_valid_ranked AS
SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY order_id_c ORDER BY order_date_p DESC, _row_hash ASC, _row_id ASC) AS dup_rank,
    FIRST_VALUE(_row_id) OVER (PARTITION BY order_id_c ORDER BY order_date_p DESC, _row_hash ASC, _row_id ASC) AS winner_row_id,
    COUNT(*) OVER (PARTITION BY order_id_c) AS group_size
FROM orders_validated
WHERE size(reasons) = 0
""")

print("Temp views orders_validated and orders_valid_ranked created.")

### Dry run: how many rows go where? (nothing written yet)

If almost everything is quarantined, a rule or list is probably wrong. Check the reason breakdown and adjust the widgets before writing.

In [0]:
preview = spark.sql("""
    SELECT
        COUNT(*)                                                        AS bronze_rows,
        SUM(CASE WHEN size(reasons) > 0 THEN 1 ELSE 0 END)              AS would_quarantine,
        SUM(CASE WHEN size(reasons) = 0 THEN 1 ELSE 0 END)              AS valid_rows
    FROM orders_validated
""").collect()[0]

quarantine_pct = preview["would_quarantine"] / preview["bronze_rows"] * 100
print(f"Bronze rows      : {preview['bronze_rows']:,}")
print(f"Would quarantine : {preview['would_quarantine']:,}  ({quarantine_pct:.2f}%)")
print(f"Valid rows       : {preview['valid_rows']:,}")

if quarantine_pct > 50:
    print("\nWARNING: more than half of the rows would be quarantined. "
          "Check the allowed lists/limits against the pre-flight output (section 3).")

In [0]:
display(spark.sql("""
    SELECT reason, COUNT(*) AS rows
    FROM orders_validated
    LATERAL VIEW explode(reasons) r AS reason
    GROUP BY reason
    ORDER BY rows DESC
"""))

## 6. Step C - write the three outcome tables

All three are rebuilt from the same Bronze snapshot. Audit identifiers (`_ingested_at`, `_source_file`, `_batch_id`, `_row_hash`, `_row_id`) are kept in every table.

In [0]:
# 6a. ACCEPTED: one deterministic winner per order_id
spark.sql(f"""
CREATE OR REPLACE TABLE {SILVER}
USING DELTA
COMMENT 'Silver: valid, typed, deduplicated orders. One row per order_id. Validate-first, then dedupe among valid rows.'
AS
SELECT
    order_id_c      AS order_id,
    customer_id_c   AS customer_id,
    order_date_p    AS order_date,
    category_c      AS category,
    quantity_p      AS quantity,
    unit_price_p    AS unit_price,
    status_c        AS order_status,
    country_c       AS country,
    _ingested_at, _source_file, _batch_id, _row_hash, _row_id,
    CAST(group_size AS INT)       AS _duplicate_group_size,
    current_timestamp()           AS _silver_processed_at
FROM orders_valid_ranked
WHERE dup_rank = 1
""")
print("silver_orders written")

In [0]:
# 6b. QUARANTINE: original raw text + every reason the row failed
spark.sql(f"""
CREATE OR REPLACE TABLE {QUARANTINE}
USING DELTA
COMMENT 'Quarantine: orders that failed at least one validation rule. Raw source values kept as text with rejection reasons.'
AS
SELECT
    order_id, customer_id, order_date, category, quantity, unit_price, order_status, country,   -- raw values
    _ingested_at, _source_file, _batch_id, _row_hash, _row_id,
    concat_ws('; ', reasons)      AS rejection_reason,
    reasons                       AS rejection_reasons,
    CAST(size(reasons) AS INT)    AS rejection_count,
    current_timestamp()           AS _silver_processed_at
FROM orders_validated
WHERE size(reasons) > 0
""")
print("quarantine_orders written")

In [0]:
# 6c. DUPLICATE LOSERS: valid rows that lost to the winner of the same order_id
spark.sql(f"""
CREATE OR REPLACE TABLE {DUP_LOSERS}
USING DELTA
COMMENT 'Valid orders that lost deterministic winner selection for their order_id. Linked to the winner by winner_row_id.'
AS
SELECT
    order_id_c      AS order_id,
    customer_id_c   AS customer_id,
    order_date_p    AS order_date,
    category_c      AS category,
    quantity_p      AS quantity,
    unit_price_p    AS unit_price,
    status_c        AS order_status,
    country_c       AS country,
    _ingested_at, _source_file, _batch_id, _row_hash, _row_id,
    winner_row_id,
    CAST(dup_rank AS INT)     AS duplicate_rank,
    CAST(group_size AS INT)   AS _duplicate_group_size,
    'latest order_date, then smallest _row_hash, then smallest _row_id' AS winner_rule,
    current_timestamp()       AS _silver_processed_at
FROM orders_valid_ranked
WHERE dup_rank > 1
""")
print("duplicate_losers_orders written")

## 7. Validation: every Bronze row has exactly one outcome

Totals matching is not enough, so this also proves **no overlap** and **nothing missing** at row level using `_row_id`. (`05_reconciliation.py` repeats this for the final report.)

In [0]:
bronze_n = spark.table(BRONZE).count()
silver_n = spark.table(SILVER).count()
quar_n = spark.table(QUARANTINE).count()
dup_n = spark.table(DUP_LOSERS).count()

print(f"Bronze rows            : {bronze_n:,}")
print(f"  accepted (silver)    : {silver_n:,}")
print(f"  quarantined          : {quar_n:,}")
print(f"  duplicate losers     : {dup_n:,}")
print(f"  accepted+quar+losers : {silver_n + quar_n + dup_n:,}")

display(spark.createDataFrame(
    [("bronze_orders", bronze_n), ("silver_orders (accepted)", silver_n),
     ("quarantine_orders", quar_n), ("duplicate_losers_orders", dup_n),
     ("accepted + quarantine + losers", silver_n + quar_n + dup_n)],
    "table STRING, rows LONG"
))

In [0]:
# Row-level proof using _row_id
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW orders_outcomes AS
SELECT _row_id, 'accepted'         AS outcome FROM {SILVER}
UNION ALL
SELECT _row_id, 'quarantine'       AS outcome FROM {QUARANTINE}
UNION ALL
SELECT _row_id, 'duplicate_loser'  AS outcome FROM {DUP_LOSERS}
""")

outcome_rows = spark.sql("SELECT COUNT(*) AS n, COUNT(DISTINCT _row_id) AS d FROM orders_outcomes").collect()[0]

missing_from_outcomes = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {BRONZE} b
    LEFT ANTI JOIN orders_outcomes o ON b._row_id = o._row_id
""").collect()[0]["n"]

unknown_in_outcomes = spark.sql(f"""
    SELECT COUNT(*) AS n FROM orders_outcomes o
    LEFT ANTI JOIN {BRONZE} b ON b._row_id = o._row_id
""").collect()[0]["n"]

dup_order_ids_in_silver = spark.sql(f"""
    SELECT COUNT(*) - COUNT(DISTINCT order_id) AS n FROM {SILVER}
""").collect()[0]["n"]

losers_without_winner = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {DUP_LOSERS} l
    LEFT ANTI JOIN {SILVER} s ON l.winner_row_id = s._row_id
""").collect()[0]["n"]

quarantine_without_reason = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {QUARANTINE}
    WHERE rejection_reason IS NULL OR trim(rejection_reason) = ''
""").collect()[0]["n"]

# Spot-check that accepted rows really satisfy the rules
rule_violations_in_silver = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {SILVER}
    WHERE order_id IS NULL OR customer_id IS NULL OR order_date IS NULL
       OR quantity IS NULL OR quantity <= 0 OR quantity > {MAX_QUANTITY}
       OR unit_price IS NULL OR unit_price < 0 OR unit_price > {MAX_UNIT_PRICE}
       OR order_date > DATE'{RUN_DATE}' OR order_date < DATE'{MIN_ORDER_DATE}'
       OR order_status NOT IN ({sql_in_list(ALLOWED_STATUSES)})
       OR category IS NULL OR country IS NULL
""").collect()[0]["n"]

In [0]:
checks = {
    "bronze = accepted + quarantine + losers": bronze_n == silver_n + quar_n + dup_n,
    "no Bronze row is missing from outcomes": missing_from_outcomes == 0,
    "no row appears in more than one outcome (distinct _row_id = outcome rows)": outcome_rows["n"] == outcome_rows["d"],
    "no outcome row is unknown to Bronze": unknown_in_outcomes == 0,
    "silver_orders has unique order_id": dup_order_ids_in_silver == 0,
    "every duplicate loser points to an accepted winner": losers_without_winner == 0,
    "every quarantined row has a rejection_reason": quarantine_without_reason == 0,
    "accepted rows satisfy all documented rules": rule_violations_in_silver == 0,
}

print("=" * 70)
for label, ok in checks.items():
    print(f"[{'PASS' if ok else 'FAIL'}] {label}")
print("=" * 70)
print("OVERALL:", "SILVER ORDERS VALID" if all(checks.values()) else "SILVER ORDERS CHECKS FAILED")

assert all(checks.values()), "Silver orders validation failed - see output above."

## 8. Evidence for `evidence/04_silver/`

In [0]:
# Rejection reasons (a row with several reasons is counted once per reason, so this can exceed the quarantine row count)
display(spark.sql(f"""
    SELECT reason AS rejection_reason, COUNT(*) AS rows_with_this_reason
    FROM {QUARANTINE}
    LATERAL VIEW explode(rejection_reasons) r AS reason
    GROUP BY reason
    ORDER BY rows_with_this_reason DESC
"""))

In [0]:
# Sample quarantined rows (raw values + reasons)
display(spark.sql(f"""
    SELECT order_id, customer_id, order_date, quantity, unit_price, order_status, rejection_reason, _row_id
    FROM {QUARANTINE}
    ORDER BY rejection_count DESC
    LIMIT 20
"""))

In [0]:
# Sample duplicate losers next to their winner, to show the deterministic rule at work
display(spark.sql(f"""
    SELECT l.order_id,
           l.duplicate_rank, l.order_date AS loser_order_date, w.order_date AS winner_order_date,
           l._row_id AS loser_row_id, w._row_id AS winner_row_id, l.winner_rule
    FROM {DUP_LOSERS} l
    JOIN {SILVER} w ON l.winner_row_id = w._row_id
    ORDER BY l.order_id, l.duplicate_rank
    LIMIT 20
"""))

In [0]:
display(spark.sql(f"DESCRIBE TABLE {SILVER}"))

In [0]:
display(spark.table(SILVER).limit(10))

## 9. README text: the rules you applied

Prints a markdown block built from the widget values actually used in this run.

In [0]:
print(f"""
## Silver orders: validation, parsing and deduplication rules

Run date used for the future-date rule: {RUN_DATE}
Bronze batch: {batches[0]}

**Precedence:** validate first, then deduplicate among valid records.

| Field | Parsing / normalization | Rule (my chosen limits, not assignment-provided) | Failure outcome |
|---|---|---|---|
| order_id | trim, blank -> NULL | not blank | quarantine |
| customer_id | trim, blank -> NULL | not blank | quarantine |
| order_date | yyyy-MM-dd and a real date -> DATE | parses; >= {MIN_ORDER_DATE}; <= run date | quarantine |
| quantity | decimal, whole number -> INT | integer, 1 .. {MAX_QUANTITY} | quarantine |
| unit_price | DECIMAL(18,2) | numeric, 0 .. {MAX_UNIT_PRICE} | quarantine |
| order_status | trim + lowercase | in {ALLOWED_STATUSES} | quarantine |
| category | trim + lowercase | not blank{'; in ' + str(ALLOWED_CATEGORIES) if ALLOWED_CATEGORIES else ' (no list enforced)'} | quarantine |
| country | trim + uppercase | not blank{'; in ' + str(ALLOWED_COUNTRIES) if ALLOWED_COUNTRIES else ' (no list enforced)'} | quarantine |
| duplicate order_id | among VALID rows only | winner = latest order_date, then smallest _row_hash, then smallest _row_id | duplicate_losers_orders |

**Outcome counts (this run):** Bronze {bronze_n:,} = accepted {silver_n:,} + quarantined {quar_n:,} + duplicate losers {dup_n:,}

**Rerun behavior:** tables are rebuilt from Bronze with CREATE OR REPLACE TABLE, so reruns do not duplicate rows.
""")

## Done

Save for `evidence/04_silver/`: the outcome-count table, the PASS summary, the rejection-reason breakdown, and a duplicate winner/loser sample.

**Next:** `04_silver_transactions.py` (validates `order_id` against the accepted `silver_orders` created here).